# Exercise 1 - Blockchain Structure and Tamper Detection


In [13]:
# Import libraries for hashing, JSON output, and timestamps.
import hashlib
import json
import time

# Return the SHA-256 hash of text.
def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Calculate one Merkle Root for all transactions.
def calculate_merkle_root(transactions):
    if not transactions:
        return sha256_text("")

    # Hash each transaction to start the Merkle tree.
    level = [sha256_text(tx) for tx in transactions]

    while len(level) > 1:
        # Duplicate the final hash when the count is odd.
        if len(level) % 2 == 1:
            level.append(level[-1])

        # Hash neighbouring pairs until one root remains.
        level = [
            sha256_text(level[i] + level[i + 1])
            for i in range(0, len(level), 2)
        ]

    return level[0]

# Store one block and its integrity information.
class Block:
    def __init__(self, index, transactions, previous_hash, timestamp=None):
        self.index = index
        self.timestamp = int(time.time()) if timestamp is None else timestamp
        self.transactions = list(transactions)
        self.previous_hash = previous_hash
        self.merkle_root = calculate_merkle_root(self.transactions)
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        # Hash the block header with SHA-256.
        header = {
            "index": self.index,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "merkle_root": self.merkle_root,
        }
        return hashlib.sha256(
            json.dumps(header, sort_keys=True).encode("utf-8")
        ).hexdigest()

    def as_dict(self):
        # Return readable block data for the output screenshots.
        return {
            "block": self.index,
            "transactions": self.transactions,
            "merkle_root": self.merkle_root,
            "previous_hash": self.previous_hash,
            "hash": self.hash,
        }


# Manage the blockchain and validate its integrity.
class Blockchain:
    def __init__(self):
        # Create the genesis block automatically.
        self.chain = [
            Block(
                index=1,
                transactions=["Genesis Block"],
                previous_hash="0" * 64,
            )
        ]

    def add_block(self, transactions):
        # Append a block linked to the current last block.
        previous_block = self.chain[-1]
        new_block = Block(
            index=len(self.chain) + 1,
            transactions=transactions,
            previous_hash=previous_block.hash,
        )
        self.chain.append(new_block)

    def validate(self):
        # Check every block and return the first failure.
        for position, block in enumerate(self.chain):
            # Check that transaction data matches the stored Merkle Root.
            recomputed_merkle = calculate_merkle_root(block.transactions)

            if block.merkle_root != recomputed_merkle:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Merkle Root mismatch",
                    "stored": block.merkle_root,
                    "recomputed": recomputed_merkle,
                    "blocks_checked": position + 1,
                }

            # Check that the stored block hash is still correct.
            recomputed_hash = block.calculate_hash()
            if block.hash != recomputed_hash:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Block hash mismatch",
                    "stored": block.hash,
                    "recomputed": recomputed_hash,
                    "blocks_checked": position + 1,
                }

            # Check the genesis rule or previous-block link.
            expected_previous = "0" * 64 if position == 0 else self.chain[position - 1].hash
            if block.previous_hash != expected_previous:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Previous-hash link mismatch",
                    "stored": block.previous_hash,
                    "recomputed": expected_previous,
                    "blocks_checked": position + 1,
                }

        return {
            "valid": True,
            "first_failing_block": None,
            "reason": "No integrity errors detected",
            "stored": None,
            "recomputed": None,
            "blocks_checked": len(self.chain),
        }

    def print_chain(self):
        # Print each block for the required evidence.
        for block in self.chain:
            print(json.dumps(block.as_dict(), indent=2))
            print("-" * 88)


In [ ]:
# Create 10 blocks: 1 genesis block plus 9 appended blocks.
blockchain = Blockchain()

# Different transaction data for Blocks 2 to 10.
transaction_sets = [
    ["Alice -> Bob: 5 BTC", "Bob -> Charlie: 1 BTC"],
    ["Charlie -> David: 2 BTC", "Eve -> Alice: 3 BTC"],
    ["David -> Alice: 0.5 BTC", "Bob -> Eve: 0.25 BTC"],
    ["Alice -> Eve: 1.2 BTC", "Charlie -> Bob: 0.8 BTC"],
    ["Eve -> David: 0.7 BTC", "David -> Bob: 0.4 BTC"],
    ["Bob -> Alice: 1.1 BTC", "Alice -> Charlie: 0.3 BTC"],
    ["Charlie -> Eve: 0.9 BTC", "Eve -> Bob: 0.2 BTC"],
    ["David -> Charlie: 1.5 BTC", "Bob -> David: 0.6 BTC"],
    ["Alice -> David: 0.75 BTC", "Eve -> Charlie: 0.45 BTC"],
]

# Add the nine non-genesis blocks.
for transactions in transaction_sets:
    blockchain.add_block(transactions)

assert len(blockchain.chain) == 10

# Print all 10 blocks in full JSON format.
print("=== BLOCKCHAIN BEFORE MODIFICATION ===")
blockchain.print_chain()

print("=== VALIDATION BEFORE MODIFICATION ===")
print(json.dumps(blockchain.validate(), indent=2))


=== BLOCKCHAIN BEFORE MODIFICATION: BLOCKS 1-5 ===
{
  "block": 1,
  "transactions": [
    "Genesis Block"
  ],
  "merkle_root": "89eb0ac031a63d2421cd05a2fbe41f3ea35f5c3712ca839cbf6b85c4ee07b7a3",
  "previous_hash": "0000000000000000000000000000000000000000000000000000000000000000",
  "hash": "1e60861ef0b215015513b447e0df12e4df5d03513624f295c93684093fed2e7f"
}
----------------------------------------------------------------------------------------
{
  "block": 2,
  "transactions": [
    "Alice -> Bob: 5 BTC",
    "Bob -> Charlie: 1 BTC"
  ],
  "merkle_root": "b45848ab11dc09d63a0b2d1299aee371827e1edee3e4c847d0c7b519af4d8f3e",
  "previous_hash": "1e60861ef0b215015513b447e0df12e4df5d03513624f295c93684093fed2e7f",
  "hash": "55adb50f26d8aa0d727ea58aa7250b70a2315abacb1dd7d43114c132ecc4aa99"
}
----------------------------------------------------------------------------------------
{
  "block": 3,
  "transactions": [
    "Charlie -> David: 2 BTC",
    "Eve -> Alice: 3 BTC"
  ],
  "merkle_root

In [11]:
# Print the remaining five blocks in full JSON format.
print("=== BLOCKCHAIN BEFORE MODIFICATION: BLOCKS 6-10 ===")
for block in blockchain.chain[5:]:
    print(json.dumps(block.as_dict(), indent=2))
    print("-" * 88)

print("=== VALIDATION BEFORE MODIFICATION ===")
print(json.dumps(blockchain.validate(), indent=2))


=== BLOCKCHAIN BEFORE MODIFICATION: BLOCKS 6-10 ===
{
  "block": 6,
  "transactions": [
    "Eve -> David: 0.7 BTC",
    "David -> Bob: 0.4 BTC"
  ],
  "merkle_root": "3b9db12db084ad84cdb30039c3d119e4a0c352177176027cbe35553479e257f4",
  "previous_hash": "3f929815cfdec97f9197176b93f2fe47379c3245a92f5b5a1521ffaf30f9718d",
  "hash": "3749d534b832543534f8196a8cc096f5163b6bfc3cd3f938b46a42b598c9a6ab"
}
----------------------------------------------------------------------------------------
{
  "block": 7,
  "transactions": [
    "Bob -> Alice: 1.1 BTC",
    "Alice -> Charlie: 0.3 BTC"
  ],
  "merkle_root": "48d95277795750adb40100601b204177358309f050ed3bf9a35760aae2c30832",
  "previous_hash": "3749d534b832543534f8196a8cc096f5163b6bfc3cd3f938b46a42b598c9a6ab",
  "hash": "b64ddd0226f15efe49ea87b0d4ecfb5998960b9b318e7ac6dd961fb438392203"
}
----------------------------------------------------------------------------------------
{
  "block": 8,
  "transactions": [
    "Charlie -> Eve: 0.9 BTC",
 

In [14]:
# Modify Block 5 without updating its Merkle Root or hash.
fifth_block = blockchain.chain[4]
original_transaction = fifth_block.transactions[0]
fifth_block.transactions[0] = "ATTACKER -> ATTACKER: 999 BTC"

print("Original transaction:", original_transaction)
print("Modified transaction:", fifth_block.transactions[0])

# Show the modified block.
print("\n=== MODIFIED 5TH BLOCK ===")
print(json.dumps(fifth_block.as_dict(), indent=2))

# The validator should identify Block 5 as the first failure.
print("\n=== VALIDATION AFTER MODIFICATION ===")
tamper_result = blockchain.validate()
print(json.dumps(tamper_result, indent=2))


Original transaction: Alice -> Eve: 1.2 BTC
Modified transaction: ATTACKER -> ATTACKER: 999 BTC

=== MODIFIED 5TH BLOCK ===
{
  "block": 5,
  "transactions": [
    "ATTACKER -> ATTACKER: 999 BTC",
    "Charlie -> Bob: 0.8 BTC"
  ],
  "merkle_root": "ac54f23535b6c68e9d79e89e0fa9e20459cb859b059683d6a4b445e5a11c3d83",
  "previous_hash": "79a2b52f7e0385dfbeda5f4b212f174dfc72e80224ecf3f70298ccff9bbee61b",
  "hash": "3f929815cfdec97f9197176b93f2fe47379c3245a92f5b5a1521ffaf30f9718d"
}

=== VALIDATION AFTER MODIFICATION ===
{
  "valid": false,
  "first_failing_block": 5,
  "reason": "Merkle Root mismatch",
  "stored": "ac54f23535b6c68e9d79e89e0fa9e20459cb859b059683d6a4b445e5a11c3d83",
  "recomputed": "635e34751f436c0550d9bed99770f49f97b8ce091fa38e68626f9a714eabece4",
  "blocks_checked": 5
}


## Reflection Questions

### 1. Why does modifying Block 5 affect later blocks?

When I changed the transaction in Block 5, its calculated Merkle Root no longer matched the Merkle Root stored in the block. The validator therefore identified Block 5 as the first invalid block. In the test, I deliberately left the original Merkle Root and hash unchanged, so the first error was reported as a Merkle Root mismatch.

If I recalculated Block 5 after changing the transaction, its hash would also change. Block 6 still contains the original Block 5 hash in its `previous_hash` field, so Block 6 would then fail the previous-hash link check. This is how the hash links make changes to earlier blocks affect the rest of the chain.

### 2. What should a validation program report?

The validation result should clearly state whether the blockchain is valid. If it is not valid, it should identify the first failing block and explain the reason for the failure. In this implementation, the result also includes the stored value, the newly calculated value, and the number of blocks checked.

These details make the result easier to understand and verify. For example, my tampering test reports that Block 5 failed because its stored Merkle Root is different from the Merkle Root recalculated from the modified transactions.

### 3. One improvement to resist tampering

One improvement would be to digitally sign each block header with the block creator's private key. The blockchain could then use the creator's public key to verify the signature during validation. An attacker might still change the transaction data locally, but they would not be able to create a valid replacement signature without the private key.

This would strengthen the current design because SHA-256 and Merkle Roots detect changes, while digital signatures also help prove that the block came from the authorised creator.
